In [1]:
import torch
import json
from pathlib import Path
from torch.utils.data import DataLoader
from torch import Tensor
from functools import partial
#locals
from src.run_types import AdminInfo, AVAIL_SETS
from src.configs import load_config
from src.video_transforms import get_temporal_augs, get_transform
from src.video_dataset import get_data_set, get_wlasl_info, VideoDataset
from src.visualise import get_all_sets, MiniSet, get_class_list
from src.models import get_model
from src.testing import setup_data, test_topk_clsrep_multiview

Please update your PyTorchVideo to latest master


In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
admin = AdminInfo.model_validate({
    "model" : "S3D",
    "dataset": "WLASL",
    "split": "asl100",
    "save_path": "runs/asl100/S3D/exp082/checkpoints000",
    "exp_no": "082",
    "recover": False,
    "config_path": "configfiles/asl100/S3D/exp082.toml",
    "weight_path": None
})
config = load_config(admin)
save_path = Path(admin.save_path)

In [4]:
set_name: AVAIL_SETS = 'test'

In [5]:
test_info = config.data.test_augs

In [6]:
assert test_info is not None
mod_t = test_info.model_copy(deep=True) 
mod_t.strict_size = False
mod_t.temporal_aug.pop(0)
mod_t.spatial_aug.pop(0)


for attr in mod_t:
    print(attr)




('normalise', True)
('norm_dict', NormDict(mean=(0.43216, 0.394666, 0.37645), std=(0.22803, 0.22145, 0.216989)))
('temporal_aug', [])
('spatial_aug', [])
('strict_size', False)
('target_length', 32)
('frame_size', 224)


In [7]:
transform, perm, sh_e = get_transform(
        norm_dict=mod_t.norm_dict,
        temporal_aug=mod_t.temporal_aug,
        spatial_aug=mod_t.spatial_aug,
        permute_time_channel=False
    )

set_info = get_wlasl_info(admin.split, set_name)

test_dataset = dataset = VideoDataset(set_info, transforms=transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

In [8]:
model = get_model(config.admin.model, test_dataset.num_classes, drop_p=0)
checkpoint = torch.load(Path(config.admin.save_path) / "best.pth", weights_only=True)
model.load_state_dict(checkpoint)

<All keys matched successfully>

In [11]:
topk_res, cls_report, all_targets, all_preds = test_topk_clsrep_multiview(
        model=model,
        test_loader=test_loader,
        verbose=False,
    )

Testing:   0%|          | 0/258 [00:00<?, ?it/s]

Testing: 100%|██████████| 258/258 [03:02<00:00,  1.41it/s]

top-k average per class acc: 0.666, 0.8891666666666667, 0.9341666666666667
top-k per instance acc: 0.6589147286821705, 0.8875968992248062, 0.937984496124031
Averag Loss: 4.06
